# SOB4ES — Pipeline de Preparación de Datos
### CRISP-ML(Q) · Fase 2: Ingeniería de Datos

> Studer et al. (2020) — [ml-ops.org/content/crisp-ml](https://ml-ops.org/content/crisp-ml)

Integra todos los conjuntos de datos SOB4ES y las fuentes raster europeas en una única tabla lista para análisis, **una fila por `SITE_ID`**.

| # | Archivo | Hoja(s) | Filas | Clave |
|---|---------|---------|-------|-------|
| 1 | DD2.2.1_SITE_DESCRIPTIONS | SITE_DETAILS | 428 | SITE_ID |
| 2 | DD2.2.3_ABIOTIC_PHYSICAL | SITE_PHYSICAL, PLOT_PHYSICAL | 428 / 1284 | SITE_ID |
| 3 | DD2.2.4_ABIOTIC_CHEMICAL | CHEM_SITE, CHEM_PLOT | 428 / 1346 | SITE_ID |
| 4 | DD2.2.5.2_ALPHA_DIVERSITIES | ALPHA_DIV_SITE, MICROBIOME_DIV | 431 / 447 | SITE_ID |
| 5 | DD2.2.6_MACROFAUNA_COM | MACROFAUNA_ORDER | 1043 parcelas | SITE_ID (agg) |
| 6 | DD2.2.7_EARTHWORMS_COM | EARTHWORM_SPECIES_ALL | 1107 parcelas | SITE_ID (agg) |
| 7 | NUID_UCD_Earthworms | Earthworms - Spatial Sampling | 690 | SITE_ID (raw) |
| 8 | UVIGO_Earthworms | Spain/Slovenia/Romania/Sweden/Israel | ~1110 | SITE_ID (raw) |
| 9 | DD2.2.8_ORIBATIDA_COM | ORIBATID_SPECIES | 358 | SITE_ID |
| 10 | DD2.2.9_MESOTIGMATA_COM | MESOSTIGMATID_SPECIES | 491 | SITE_ID |
| 11 | DD2.2.10_COLLEMBOLA_COM | COLLEMBOLA_SPECIES | 362 | SITE_ID |
| 12 | DD2.2.12_BACTERIA_SEQ | DD2.2.12_BACTERIA_SEQ | 447 × 6798 ASVs | SAMPLE_ID |
| 13 | DD2.2.13_FUNGI_SEQ | DD2.2.13_FUNGI_SEQ | 445 × 14 ASVs | SAMPLE_ID |
| 14 | DD2.2.14_EUKARYOTE_SEQ | DD2.2.14_18S_SEQ | 447 × 4077 ASVs | SAMPLE_ID |
| 15 | DD2.2.15_OOMYCETES_CERCOZOA_SEQ | OOMYCETE_SEQ, CERCOZOAN_SEQ | 457 | SITE_ID |

**Capas raster EU** (extracción punto a punto por coordenadas):

| Capa | Archivo | Resolución | Fuente |
|------|---------|------------|--------|
| Densidad aparente | bulk_density.tif | 500 m | ESDAC/LUCAS |
| Arcilla / Arena / Limo | clay/sand/silt_content.tif | 500 m | ESDAC/LUCAS |
| Textura USDA | soil_texture.tif | 500 m | ESDAC/LUCAS |
| Capacidad de retención hídrica | water_holding_capacity.tif | 500 m | ESDAC/LUCAS |
| C:N, K, N, P, pH | CN/K/N/P/pH.tif | 500 m | ESDAC/LUCAS |
| Carbono orgánico (LUCAS) | LUCAS-median.tif | 250 m | ESDAC/LUCAS |
| Carbono orgánico (OCTOP) | octop_insp.tif | 1000 m | ESDAC/OCTOP |
| Cobre | copper_map_fill.tif | 500 m | ESDAC |
| Níquel / Plomo | Ni_EU27.tif / Pb_EU27.tif | 1000 m | ESDAC/LUCAS 2009 |
| Zinc | zinc.tif | 1000 m | ESDAC/LUCAS 2009 |
| Zonas ambientales | eu_env_zones_2018_esdac.tif | 100 m | EEA 2018 |
| Uso del suelo | eu_land_cover_2018_corine.tif | 100 m | CORINE 2018 |
| Tipo de suelo WRB | eu_soil_type_wrb_2006_esdac.tif | 1000 m | ESDAC 2006 |

**Estrategia de unión**: todas las tablas se unen por `SITE_ID`. Las tablas a nivel de parcela se agregan (media) a nivel de sitio. Los datos de secuenciación (bacteria, eucariotas) se resumen como riqueza + lecturas totales para evitar tablas de ~10 000 columnas.


---
## 0 · Configuración del Entorno


In [67]:
import os
import warnings
import numpy  as np
import pandas as pd
import openpyxl
import rasterio
from rasterio.crs       import CRS
from rasterio.warp      import transform as rio_transform
from rasterio.transform import rowcol
from dbfread            import DBF
from tqdm.notebook      import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

# Rutas de datos
DATA_DIR = '../Datasets/SOB4ES_DATASETS/'     # datos SOB4ES
EU_DIR   = '../Datasets/EU_DATASETS/'     # rasters europeos (misma carpeta)
OUT_DIR  = 'output/'
os.makedirs(OUT_DIR, exist_ok=True)

WGS84 = CRS.from_epsg(4326)  # CRS de las coordenadas SOB4ES

print('Entorno configurado...')


Entorno configurado...


---
## 1 · Descripción de Sitios *(tabla base)*

`SITE_DETAILS` es la tabla base: 428 sitios, una fila cada uno.
Todos los demás conjuntos de datos se unirán a esta tabla mediante `SITE_ID`.


In [68]:
sites = pd.read_excel(
    DATA_DIR + 'DD2.2.1_SITE_DESCRIPTIONS.xlsx',
    sheet_name='SITE_DETAILS'
)

# Renombrar columnas para mayor claridad
sites = sites.rename(columns={
    'Site_latitude'   : 'latitude',
    'Site_longitude'  : 'longitude',
    'Soil_type_WRB'   : 'soil_type',
})

# Parsear fecha de muestreo correctamente
sites['Sampling_date'] = pd.to_datetime(sites['Sampling_date'], errors='coerce')

print(f'Sites: {sites.shape}  |  unique SITE_IDs: {sites["SITE_ID"].nunique()}')
print(f'Countries: {sorted(sites["Country"].unique())}')
sites.head(3)

Sites: (428, 13)  |  unique SITE_IDs: 428
Countries: ['BE', 'CH', 'DE', 'ES', 'FR', 'IE', 'IL', 'IT', 'NL', 'RO', 'SE', 'SI']


,SITE_ID,SAMPLE_ID,Country,Pedoclimatic_region,Site_locality,latitude,longitude,Sampling_date,soil_type,Land_use_type,Land_use_intensity,Dominant_vegetation,Total_Plant_cover
0,BE_001,BE_001_DESCRIPTION,BE,ATC,Geel,51.0990,4.9750,2023-10-24,Umbrisol,Grassland,Low,Grass,100.0000
1,BE_002,BE_002_DESCRIPTION,BE,ATC,Geel,NaN,NaN,2023-10-24,Umbrisol,Grassland,Mid,Grass,100.0000
2,BE_003,BE_003_DESCRIPTION,BE,ATC,Geel,51.0970,4.9760,2023-10-27,Arenosol,Grassland,High,Grass,100.0000


---
## 2 · Datos Abióticos Físicos

Dos hojas:
- `SITE_PHYSICAL` — una fila por sitio (arcilla, limo, arena, densidad aparente, humedad, estabilidad de agregados)
- `PLOT_PHYSICAL` — tres parcelas por sitio (A/B/C) con columna `SOIL_LAYER` → se agrega a la media del sitio


In [69]:
# Datos físicos a nivel de sitio 
phys_site = pd.read_excel(
    DATA_DIR + 'DD2.2.3_ABIOTIC_PHYSICAL.xlsx',
    sheet_name='DD2.2.3_SITE_PHYSICAL'
).drop(columns=['SAMPLE_ID'])   # SITE_ID is the join key

print(f'PHYS_SITE: {phys_site.shape}')
phys_site.head(2)

PHYS_SITE: (428, 7)


,SITE_ID,clay_content,silt_content,sand_content,aggregate_stability,Bulk density,Soil moisture
0,BE_001,28.5490,35.3960,36.0550,0.9270,0.4233,1.4733
1,BE_002,24.5190,28.9250,46.5560,0.8830,0.8033,0.8400


In [70]:
# Datos físicos a nivel de parcela → media por sitio
phys_plot = pd.read_excel(
    DATA_DIR + 'DD2.2.3_ABIOTIC_PHYSICAL.xlsx',
    sheet_name='DD2.2.3_PLOT_PHYSICAL'
)

phys_plot_agg = (
    phys_plot
    .drop(columns=['PLOT_ID', 'SAMPLE_ID'])
    .groupby(['SITE_ID', 'SOIL_LAYER'])
    .mean(numeric_only=True)
    .reset_index()
)

# Pivotear para que cada SOIL_LAYER (M=mineral, O=orgánica) sea un sufijo de columna
phys_plot_pivot = phys_plot_agg.pivot(index='SITE_ID', columns='SOIL_LAYER').round(6)
phys_plot_pivot.columns = [f'plot_{col}_{layer}' for col, layer in phys_plot_pivot.columns]
phys_plot_pivot = phys_plot_pivot.reset_index()

print(f'PHYS_PLOT pivoted: {phys_plot_pivot.shape}')
phys_plot_pivot.head(2)

PHYS_PLOT pivoted: (428, 6)


,SITE_ID,plot_clay_content_M,plot_silt_content_M,plot_sand_content_M,plot_aggregate_stability_M,plot_Bulk density_M
0,BE_001,28.5490,35.3960,36.0550,0.9270,0.4265
1,BE_002,24.5190,28.9250,46.5560,0.8830,0.8060


---
## 3 · Datos Abióticos Químicos

- `CHEM_SITE` — metales pesados + pH por sitio
- `CHEM_PLOT` — Total_C, Total_organic_C, Total_N por parcela → se agrega a la media del sitio


In [71]:
# Química a nivel de sitio
chem_site = pd.read_excel(
    DATA_DIR + 'DD2.2.4_ABIOTIC_CHEMICAL.xlsx',
    sheet_name='DD2.2.4_CHEM_SITE'
).drop(columns=['SAMPLE_ID'])

print(f'CHEM_SITE: {chem_site.shape}')
print(f'Columns: {list(chem_site.columns)}')
chem_site.head(2)

CHEM_SITE: (428, 10)
Columns: ['SITE_ID', 'As', 'Cu', 'K', 'Mo', 'Ni', 'P', 'Pb', 'Zn', 'soil_pH']


,SITE_ID,As,Cu,K,Mo,Ni,P,Pb,Zn,soil_pH
0,BE_001,159.0000,15.0000,8.7137,0.0000,0,6.5939,93.0000,129.0000,4.5000
1,BE_002,53.3000,14.4000,7.5104,4.0000,0,7.0087,47.3000,79.0000,4.9100


In [72]:
# Química a nivel de parcela → media por sitio 
chem_plot = pd.read_excel(
    DATA_DIR + 'DD2.2.4_ABIOTIC_CHEMICAL.xlsx',
    sheet_name='DD_2.4_CHEM_PLOT'
)

chem_plot_agg = (
    chem_plot
    .drop(columns=['PLOT_ID'])
    .groupby('SITE_ID')
    .mean(numeric_only=True)
    .add_prefix('plot_')
    .reset_index()
)

print(f'CHEM_PLOT aggregated: {chem_plot_agg.shape}')
chem_plot_agg.head(2)

CHEM_PLOT aggregated: (428, 4)


,SITE_ID,plot_Total_C,plot_Total_organic_C,plot_Total_N
0,BE_001,12.8458,12.8458,0.9610
1,BE_002,10.6648,10.6648,0.8655


---
## 4 · Índices de Diversidad Alfa

Dos hojas con nombres de columna clave distintos:
- `ALPHA_DIV_SITE` usa `SITE_ID`
- `MICROBIOME_DIV_SITE` usa `SampleID` (mismos valores que SITE_ID — se renombra)


In [73]:
# Diversidades alfa de macrofauna, lombrices y ácaros 
alpha_div = pd.read_excel(
    DATA_DIR + 'DD2.2.5.2_ALPHA_DIVERSITIES.xlsx',
    sheet_name='ALPHA_DIV_SITE'
)

# Diversidades alfa microbianas 
micro_div = pd.read_excel(
    DATA_DIR + 'DD2.2.5.2_ALPHA_DIVERSITIES.xlsx',
    sheet_name='MICROBIOME_DIV_SITE'
).rename(columns={'SampleID': 'SITE_ID'})   # harmonise key name

print(f'ALPHA_DIV: {alpha_div.shape}  |  MICRO_DIV: {micro_div.shape}')
print('Alpha cols:', list(alpha_div.columns))
print('Micro cols:', list(micro_div.columns))

ALPHA_DIV: (431, 9)  |  MICRO_DIV: (447, 4)
Alpha cols: ['SITE_ID', 'Macrofauna_Shannon', 'Earthworm_Shannon', 'Oribatid_Shannon', 'Mesostigmatid_Shannon', 'Collembola_Shannon', 'Oomycete_Shannon', 'Cercozoan_Shannon', 'NEMATODE_Shannon']
Micro cols: ['SITE_ID', 'BACTERIA_SHANNON', 'FUNGI_SHANNON', 'EUKARYOTES_SHANNON']


---
## 5 · Comunidad de Macrofauna

Conteos a nivel de parcela por orden → se agrega por sitio: abundancia media por orden + abundancia total + riqueza de órdenes.


In [74]:
macro = pd.read_excel(
    DATA_DIR + 'DD2.2.6_MACROFAUNA_COM.xlsx',
    sheet_name='MACROFAUNA_ORDER'
)

macro_taxa_cols = [c for c in macro.columns if c not in ['SITE_ID', 'PLOT_ID', 'SAMPLE_ID']]

macro_agg = (
    macro
    .drop(columns=['PLOT_ID', 'SAMPLE_ID'])
    .groupby('SITE_ID')[macro_taxa_cols]
    .mean(numeric_only=True)
    .add_prefix('macro_')
)

# Summary features
macro_agg['macro_total_abundance'] = macro_agg.sum(axis=1)
macro_agg['macro_order_richness']  = (macro_agg[macro_agg.columns[:-1]] > 0).sum(axis=1)
macro_agg = macro_agg.reset_index()

print(f'MACROFAUNA aggregated: {macro_agg.shape}')
macro_agg.head(2)

MACROFAUNA aggregated: (368, 3)


,SITE_ID,macro_total_abundance,macro_order_richness
0,BE_001,0.0000,0.0000
1,BE_002,0.0000,0.0000


---
## 6 · Comunidad de Lombrices

### 6.1 Archivo combinado (`DD2.2.7_EARTHWORMS_COM`)

112 especies × 1107 filas de parcelas → se agrega por sitio.


In [75]:
ew_com = pd.read_excel(
    DATA_DIR + 'DD2.2.7_EARTHWORMS_COM.xlsx',
    sheet_name='EARTHWORM_SPECIES_ALL'
)

ew_species_cols = [c for c in ew_com.columns if c not in ['SITE_ID', 'PLOT_ID', 'SAMPLE_ID']]

ew_com_agg = (
    ew_com
    .drop(columns=['PLOT_ID', 'SAMPLE_ID'])
    .groupby('SITE_ID')[ew_species_cols]
    .mean(numeric_only=True)
    .add_prefix('ew_')
)

ew_com_agg['ew_total_abundance'] = ew_com_agg.sum(axis=1)
ew_com_agg['ew_species_richness'] = (ew_com_agg[ew_com_agg.columns[:-1]] > 0).sum(axis=1)
ew_com_agg = ew_com_agg.reset_index()

print(f'EARTHWORMS aggregated: {ew_com_agg.shape}')
print(f'Sites with earthworm data: {ew_com_agg["SITE_ID"].nunique()}')

EARTHWORMS aggregated: (384, 115)
Sites with earthworm data: 384


### 6.2 Archivos Raw de Lombrices — Validación

**Nota:** Según la documentación del dataset, el archivo combinado puede no agregar correctamente los conteos raw.
Aquí se reconcilian calculando la abundancia total a partir de los archivos raw y comparándola.


In [76]:
# Archivo raw NUID_UCD 
# Fila 0: Cabecera de los grupos (Juveniles / Adultos), fila 1: nombres de especies, fila 2+: datos
nuid_raw = pd.read_excel(
    DATA_DIR + 'EARTHWORMS_RAW/NUID_UCD_Earthworms.xlsx',
    sheet_name='Earthworms - Spatial Sampling',
    header=None
)

# Los nombres reales de columna están en la fila 1; la fila 0 es cabecera de grupo -> eliminar
nuid_raw.columns = ['SAMPLE_ID'] + nuid_raw.iloc[1, 1:].tolist()
nuid_raw = nuid_raw.iloc[2:].reset_index(drop=True)  

# Extraer SITE_ID a partir de SAMPLE_ID (e.g. 'IE_001_A_M_W' → 'IE_001')
nuid_raw['SITE_ID'] = nuid_raw['SAMPLE_ID'].str.extract(r'^([A-Z]{2}_\d{3})')

# Conservar solo columnas numéricas; convertir a float
nuid_num = nuid_raw.drop(columns=['SAMPLE_ID'])
for c in nuid_num.columns:
    if c != 'SITE_ID':
        nuid_num[c] = pd.to_numeric(nuid_num[c], errors='coerce')

nuid_site = (
    nuid_num
    .groupby('SITE_ID')
    .sum(numeric_only=True)
    .add_prefix('nuid_')
    .reset_index()
)
nuid_site['nuid_total_abundance'] = nuid_site.iloc[:, 1:].sum(axis=1)

print(f'NUID_UCD: {nuid_site.shape}  |  Countries: {nuid_site["SITE_ID"].str[:2].unique().tolist()}')

NUID_UCD: (218, 29)  |  Countries: ['BE', 'CH', 'DE', 'FR', 'IE', 'NL']


In [77]:
# Archivo raw UVIGO — 5 hojas por país 
uvigo_sheets = ['Spain', 'Slovenia', 'Romania', 'Sweden', 'Israel']
uvigo_frames = []

for sheet in uvigo_sheets:
    df = pd.read_excel(
        DATA_DIR + 'EARTHWORMS_RAW/UVIGO_Earthworms.xlsx',
        sheet_name=sheet
    )
    df['_sheet'] = sheet
    uvigo_frames.append(df)

uvigo_all = pd.concat(uvigo_frames, ignore_index=True)

# Extract SITE_ID from 'Sample ID' (e.g. 'ES_001_A' → 'ES_001')
uvigo_all['SITE_ID'] = uvigo_all['Sample ID'].str.extract(r'^([A-Z]{2}_\d{3})')

# Keep summary numeric columns
uvigo_keep = ['SITE_ID', 'Total No. Individuals', 'Total density', 'Total biomass per m2', 'Species richness']
# 'Total biomass per m2 per plot' exists in some sheets — normalise column name
uvigo_all = uvigo_all.rename(columns={
    'Total biomass per m2 per plot' : 'Total biomass per m2',
    'Total density per plot'        : 'Total density',
})

uvigo_site = (
    uvigo_all[uvigo_keep]
    .groupby('SITE_ID')
    .mean(numeric_only=True)
    .add_prefix('uvigo_')
    .reset_index()
)

print(f'UVIGO: {uvigo_site.shape}  |  Countries: {uvigo_site["SITE_ID"].str[:2].unique().tolist()}')

UVIGO: (162, 6)  |  Countries: ['ES', 'RO', 'SE', 'SL']


In [78]:
# Verificación cruzada: abundancia total combinada vs raw
check = (
    ew_com_agg[['SITE_ID', 'ew_total_abundance']]
    .merge(nuid_site[['SITE_ID', 'nuid_total_abundance']], on='SITE_ID', how='outer')
    .merge(uvigo_site[['SITE_ID', 'uvigo_Total No. Individuals']], on='SITE_ID', how='outer')
)

# Sitios donde el combinado tiene datos pero el raw no (o viceversa)
combined_only = check[check['nuid_total_abundance'].isna() & check['uvigo_Total No. Individuals'].isna()]
raw_only      = check[check['ew_total_abundance'].isna()]

print(f'Sites in combined only (no raw match)  : {len(combined_only)}')
print(f'Sites in raw only (missing from combined): {len(raw_only)}')
print()
print('Sample of mismatches (first 5):')
mismatch = check.dropna().copy()
mismatch['delta_nuid'] = abs(mismatch['ew_total_abundance'] - mismatch['nuid_total_abundance'])
print(mismatch[mismatch['delta_nuid'] > 0.01][['SITE_ID','ew_total_abundance','nuid_total_abundance','delta_nuid']].head())

Sites in combined only (no raw match)  : 86
Sites in raw only (missing from combined): 82

Sample of mismatches (first 5):
Empty DataFrame
Columns: [SITE_ID, ew_total_abundance, nuid_total_abundance, delta_nuid]
Index: []


---
## 7 · Comunidades de Ácaros del Suelo y Colémbolos

Los tres archivos tienen columna `SOIL_LAYER` (`M` = mineral, `O` = orgánica).
Estrategia: se usa solo la capa mineral (`M`) para una comparación consistente entre sitios,
y se resume como riqueza + abundancia total (las columnas de especies son muy anchas para unirlas directamente:
Oribátida tiene 329 spp., Mesostigmata 205, Colémbolos 115).


In [79]:
def summarise_community(filepath, sheet, id_cols, layer_col=None, prefix='', layer='M'):
    """
    Load a community abundance sheet and return a site-level summary:
    - total abundance (sum of all species counts)
    - species richness (number of species present)
    - mean abundance per species (diversity signal)
    Optionally filters to a single soil layer.
    """
    df = pd.read_excel(filepath, sheet_name=sheet)

    if layer_col and layer_col in df.columns:
        df = df[df[layer_col] == layer]

    species_cols = [c for c in df.columns if c not in id_cols + [layer_col]]
    species_cols = [c for c in species_cols if c is not None]

    df[species_cols] = df[species_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

    summary = df.groupby('SITE_ID')[species_cols].mean(numeric_only=True)
    summary[prefix + 'total_abundance'] = summary.sum(axis=1)
    summary[prefix + 'species_richness'] = (summary[species_cols] > 0).sum(axis=1)

    return summary[[prefix + 'total_abundance', prefix + 'species_richness']].reset_index()


# Oribátida (capa mineral)
orib_sum = summarise_community(
    DATA_DIR + 'DD2.2.8_ORIBATIDA_COM.xlsx',
    sheet='ORIBATID_SPECIES',
    id_cols=['SITE_ID', 'SAMPLE_ID'], layer_col='SOIL_LAYER',
    prefix='orib_', layer='M'
)

# Mesostigmata (capa mineral)
meso_sum = summarise_community(
    DATA_DIR + 'DD2.2.9_MESOTIGMATA_COM.xlsx',
    sheet='MESOSTIGMATID_SPECIES',
    id_cols=['SITE_ID', 'SAMPLE_ID'], layer_col='SOIL LAYER',
    prefix='meso_', layer='M'
)

# Colémbolos (capa mineral)
coll_sum = summarise_community(
    DATA_DIR + 'DD2.2.10_COLLEMBOLA_COM.xlsx',
    sheet='COLLEMBOLA_SPECIES',
    id_cols=['SITE_ID', 'SAMPLE_ID'], layer_col='Soil Layer',
    prefix='coll_', layer='M'
)

print(f'Oribatida: {orib_sum.shape}  |  Mesostigmata: {meso_sum.shape}  |  Collembola: {coll_sum.shape}')

Oribatida: (308, 3)  |  Mesostigmata: (433, 3)  |  Collembola: (334, 3)


---
## 8 · Datos de Secuenciación (Bacterias, Hongos, Eucariotas, Oomycetes, Cercozoa)

Las tablas ASV crudas son muy anchas (bacterias: 6798 ASVs, eucariotas: 4077 ASVs).
Se calculan características de resumen por muestra:
- **Riqueza de ASVs** — número de ASVs con al menos 1 lectura
- **Lecturas totales** — profundidad de secuenciación total

Las tablas completas de ASVs se mantienen separadas por si se necesitan en análisis posteriores.


In [80]:
def summarise_seq(filepath, sheet, sample_col='SAMPLE_ID', prefix=''):
    """
    Read a sequencing ASV table and return per-sample richness + total reads.
    The sample_col is harmonised to SITE_ID for joining (strip suffix after 3rd _).
    """
    df = pd.read_excel(filepath, sheet_name=sheet)
    asv_cols = [c for c in df.columns if c != sample_col]

    df[asv_cols] = df[asv_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    df[prefix + 'asv_richness']  = (df[asv_cols] > 0).sum(axis=1)
    df[prefix + 'total_reads']   = df[asv_cols].sum(axis=1)

    # SAMPLE_ID → SITE_ID  (e.g. 'BE_002_BAC' or just 'BE_002' → 'BE_002')
    df['SITE_ID'] = df[sample_col].str.extract(r'^([A-Z]{2}_\d{3})')

    return (
        df.groupby('SITE_ID')[[prefix + 'asv_richness', prefix + 'total_reads']]
          .mean(numeric_only=True)
          .reset_index()
    )


bac_sum  = summarise_seq(DATA_DIR + 'DD2.2.12_BACTERIA_SEQ.xlsx',           'DD2.2.12_BACTERIA_SEQ', prefix='bac_')
fun_sum  = summarise_seq(DATA_DIR + 'DD2.2.13_FUNGI_SEQ.xlsx',              'DD2.2.13_FUNGI_SEQ',    prefix='fun_')
euk_sum  = summarise_seq(DATA_DIR + 'DD2.2.14_EUKARYOTE_SEQ.xlsx',          'DD2.2.14_18S_SEQ',      prefix='euk_')

# Oomycetes & Cercozoa have SITE_ID directly
oomy_sum = summarise_seq(DATA_DIR + 'DD2.2.15_OOMYCETES_CERCOZOA_SEQ.xlsx', 'OOMYCETE_SEQ',           prefix='oomy_')
cerc_sum = summarise_seq(DATA_DIR + 'DD2.2.15_OOMYCETES_CERCOZOA_SEQ.xlsx', 'CERCOZOAN_SEQ',          prefix='cerc_')

for name, df in [('Bacteria', bac_sum), ('Fungi', fun_sum), ('Eukaryotes', euk_sum),
                  ('Oomycetes', oomy_sum), ('Cercozoa', cerc_sum)]:
    print(f'{name:12s}: {df.shape}  sites: {df["SITE_ID"].nunique()}')

Bacteria    : (391, 3)  sites: 391
Fungi       : (390, 3)  sites: 390
Eukaryotes  : (391, 3)  sites: 391
Oomycetes   : (398, 3)  sites: 398
Cercozoa    : (398, 3)  sites: 398


---
## 9 · Características Raster Europeas (Extracción de Valores por Punto)

Todos los rasters están en el estándar **ETRS89-LAEA**, el cual es equivalente a EPSG:3035.
Para cada sitio SOB4ES `(lat, lon)` en WGS84, el punto se reproyecta al CRS nativo del raster
y se extrae el valor del píxel correspondiente.

| Tipo | Capas | Resolución | Nodata |
|------|-------|------------|--------|
| Físicas | bulk_density, clay/sand/silt, textura, WHC | 500 m | −3.4×10³⁸ (float32) |
| Químicas | CN, K, N, P, pH | 500 m | −3.4×10³⁸ (float32) |
| Arsénico | LUCAS-median.tif | 250 m | −3.4×10³⁸ (float32) |
| Carbono orgánico OCTOP | octop_insp.tif | 1000 m | −3.4×10³⁸ (float32) |
| Metales pesados | Cu (500 m) · Ni, Pb, Zn (1000 m) | mixto | −3.4×10³⁸ (float32) |
| Zonas ambientales | eu_env_zones_2018_esdac.tif | 100 m | 0 (uint8) |
| Uso del suelo | eu_land_cover_2018_corine.tif | 100 m | −128 (int8) |
| Tipo de suelo WRB | eu_soil_type_wrb_2006_esdac.tif | 1000 m | 0 (uint8) |

### Nota sobre el dataset OCTOP
Los archivos `arc.dir`, `arc0000.dat`, `arc0000.nit`, `arc0001.dat`, `arc0001.nit`,
`dblbnd.adf`, `hdr.adf`, `sta.adf`, `w001001.adf`, `w001001x.adf` y `metadata.xml`
son el **formato ESRI GRID original** del dataset OCTOP (Carbono Orgánico en el Horizonte
Superior de Suelos en Europa, ESDAC). El archivo `octop_insp.tif` es la conversión a GeoTIFF
con el CRS correctamente embebido (EPSG:3035). Ambos dan valores idénticos, por lo tanto **se usa el TIF**.

### Nota sobre el ESRI GRID sin CRS embebido
El `hdr.adf` abre correctamente con rasterio pero devuelve `CRS=None`.
El `metadata.xml` confirma que la proyección es `ETRS_1989_LAEA` (= EPSG:3035).
En caso de querer el GRID directamente, se puede obtener mediante la asignación directa del CRS de la siguiente forma:

```python
with rasterio.open('hdr.adf') as src:
    crs = CRS.from_epsg(3035)   # asignado manualmente desde metadata.xml
    xs, ys = rio_transform(WGS84, crs, [lon], [lat])
```


In [81]:
# Helpers de extracción raster
#
# Valores nodata en los archivos EU:
#   float32  → -3.4028e+38  (o -3.4000e+38 en LUCAS-median)
#   int32    → -2147483648  (soil_texture.tif)
#   uint8    → 0            (env_zones, soil_type WRB  — 0 no es clase válida)
#   int8     → -128         (CORINE land cover)

def _apply_nodata_mask(vals: np.ndarray, nodata) -> np.ndarray:
    """Convierte nodata y el centinela float32 a NaN en un array."""
    vals = vals.astype(float)
    if nodata is not None:
        vals[np.isclose(vals, float(nodata), rtol=1e-3)] = np.nan
    vals[vals < -1e35] = np.nan   # centinela float32
    return vals


def batch_query_raster(tif_path: str,
                       lats: np.ndarray,
                       lons: np.ndarray) -> np.ndarray:
    """
    Extrae los valores de un GeoTIFF para N puntos (lat, lon) en WGS84.
    Abre el archivo UNA SOLA VEZ y usa rasterio.sample() para leer
    solo los píxeles necesarios — mucho más eficiente que un bucle por fila.

    Devuelve un array float64 de longitud N; NaN en puntos nodata/fuera de bounds.
    """
    with rasterio.open(tif_path) as src:
        # Reprojectar todas las coords de golpe al CRS nativo del raster
        xs, ys = rio_transform(WGS84, src.crs, lons.tolist(), lats.tolist())

        # rasterio.sample() lee solo los tiles necesarios (eficiente en COG/GeoTIFF)
        # Devuelve un generador de arrays de 1 elemento por punto
        sampled = np.array(
            [v[0] for v in src.sample(zip(xs, ys), indexes=1, masked=False)],
            dtype=float
        )

        # Marcar puntos fuera de la extensión del raster como NaN
        bounds = src.bounds
        out_of_bounds = (
            (np.array(xs) < bounds.left)  | (np.array(xs) > bounds.right) |
            (np.array(ys) < bounds.bottom)| (np.array(ys) > bounds.top)
        )
        sampled[out_of_bounds] = np.nan

        return _apply_nodata_mask(sampled, src.nodata)


# Diccionarios de lookup (código entero → etiqueta textual)

# 1. Clases de textura USDA (soil_texture.tif, códigos 1-12)
USDA_TEXTURE = {
    1:'Clay',               2:'Silty Clay',     3:'Silty Clay Loam',    4:'Sandy Clay',
    5:'Sandy Clay Loam',    6:'Clay Loam',      7:'Silt',               8:'Silt Loam',
    9:'Sandy Loam',         10:'Loamy Sand',    11:'Sand',              12:'Loam',
}
dbf_tex_codes = {int(r['Value']) for r in DBF(EU_DIR + 'eu_2015_esdac/soil_texture.vat.dbf')}
for code in dbf_tex_codes - set(USDA_TEXTURE):
    USDA_TEXTURE[code] = f'Class_{code}'
print(f'Textura USDA: {len(USDA_TEXTURE)} clases')


# 2. Zonas Ambientales EEA 2018 (códigos 1-15, nodata=0)
#EEA_ENV_ZONES (Nombre completo)
#    1:'Alpine North',          2:'Boreal',                     3:'Nemoral',
#    4:'Atlantic North',        5:'Alpine South',               6:'Continental',
#    7:'Atlantic Cental',       8:'Pannonian',                  9:'Lusitanian',
#    10:'Anatolian',            11:'Mediterranean Mountains',   12:'Mediterranean North',
#    13:'Mediterranean South',  14:'Macaronesia',               15:'Arctic'
#

EEA_ENV_ZONES = {
    1:'ALN',   2:'BOR',    3:'NEM',
    4:'ALN',   5:'ALS',    6:'CON',
    7:'ATC',   8:'PAN',    9:'LUS',
    10:'ANA',  11:'MDM',   12:'MDN',
    13:'MDS',  14:'MAC',   15:'ARC'
}
print(f'Zonas EEA: {len(EEA_ENV_ZONES)} zonas')


# 3. CORINE Land Cover 2018 (nodata=-128; leyenda sin cabecera — fila 0 = dato)
_corine_df = pd.read_csv(
    EU_DIR + 'eu_land_cover_2018_corine/eu_land_cover_2018_corine_legend.csv',
    header=None, names=['code','r','g','b','alpha','label']
)
CORINE_LABELS = dict(zip(_corine_df['code'].astype(int), _corine_df['label']))
print(f'CORINE: {len(CORINE_LABELS)} clases (rango: {min(CORINE_LABELS)}–{max(CORINE_LABELS)})')


# 4. Tipo de Suelo WRB 2006 (códigos 1-30, nodata=0)
#    Valores 1-6 = no-suelo (Urbano, Agua…); 7-30 = Grupos de Referencia WRB
_wrb_vat = pd.DataFrame(iter(DBF(EU_DIR + 'eu_soil_type_wrb_2006/eu_soil_type_wrb_2006_esdac.vat.dbf')))
_wrb_leg = pd.read_csv(EU_DIR + 'eu_soil_type_wrb_2006/eu_soil_type_wrb_2006_esdac_legend.csv')
_wrb_joined = _wrb_vat.merge(_wrb_leg, left_on='WRBLV1', right_on='code', how='left')
NON_SOIL_WRB = {1:'Town', 2:'Soil disturbed by man', 3:'Water body',
                4:'Marsh', 5:'Glacier', 6:'Rock outcrops', 30:'No information'}
WRB_SOIL_LABELS = {}
for _, row in _wrb_joined.iterrows():
    val = int(row['VALUE'])
    WRB_SOIL_LABELS[val] = (
        NON_SOIL_WRB[val] if val in NON_SOIL_WRB
        else f"{row['WRBLV1']} – {row['description']}" if pd.notna(row.get('description'))
        else str(row['WRBLV1'])
    )
print(f'WRB suelo: {len(WRB_SOIL_LABELS)} clases')


Textura USDA: 12 clases
Zonas EEA: 15 zonas
CORINE: 45 clases (rango: 111–999)
WRB suelo: 30 clases


In [82]:
# Registro de Capas Raster Europeas
# Formato: 'nombre_columna': (ruta_archivo, unidad, (límite_inf, límite_sup))
# Para añadir una nueva capa: agregar una línea y volver a ejecutar las celdas 28-31.
# Si el límite no tiene restricción, usar None.

EU_RASTERS = {
    # Propiedades físicas — 500 m (eu_2015_esdac)

    'eu_bulk_density'           : (EU_DIR + 'eu_2015_esdac/bulk_density.tif',                   'g/cm³',  (0.0,   3.0)),
    'eu_clay_content'           : (EU_DIR + 'eu_2015_esdac/clay_content.tif',                   '%',      (0.0, 100.0)),
    'eu_sand_content'           : (EU_DIR + 'eu_2015_esdac/sand_content.tif',                   '%',      (0.0, 100.0)),
    'eu_silt_content'           : (EU_DIR + 'eu_2015_esdac/silt_content.tif',                   '%',      (0.0, 100.0)),
    'eu_soil_texture_class'     : (EU_DIR + 'eu_2015_esdac/soil_texture.tif',                   'USDA',   (1,      12)),
    'eu_water_holding_capacity' : (EU_DIR + 'eu_2015_esdac/water_holding_capacity.tif',         'vol fr', (0.0,  None)),
    
    # Propiedades químicas — 500 m (eu_2019_chemical_esdac)
    
    'eu_CN_ratio'               : (EU_DIR + 'eu_2019_chemical_esdac/CN.tif',                    'ratio',  (0.0,  None)),
    'eu_K'                      : (EU_DIR + 'eu_2019_chemical_esdac/K.tif',                     'mg/kg',  (0.0,  None)),
    'eu_N'                      : (EU_DIR + 'eu_2019_chemical_esdac/N.tif',                     'g/kg',   (0.0,  None)),
    'eu_P'                      : (EU_DIR + 'eu_2019_chemical_esdac/P.tif',                     'mg/kg',  (0.0,  None)),
    'eu_pH'                     : (EU_DIR + 'eu_2019_chemical_esdac/pH.tif',                    'pH',     (0.0,  14.0)),
    
    # Arsénico — 250 m (eu_arsenic)
    
    'eu_As'   : (EU_DIR + 'eu_arsenic/LUCAS-median.tif',                      '%',      (0.0, 100.0)),
    
    # Carbono orgánico — 1000 m (octop_insp_directory)
    # Archivo fuente original: ESRI GRID (hdr.adf + w001001.adf + resto de .adf)
    # Se usa el GeoTIFF exportado (octop_insp.tif) porque tiene CRS embebido.
    # Ambos producen valores idénticos (verificado en BE_001: 3.2933%).
    
    'eu_organic_carbon_octop'   : (EU_DIR + 'octop_insp_directory/octop_insp.tif',              '%',      (0.0, 100.0)),
    
    # Metales pesados — 500 m (eu_copper)
    
    'eu_Cu'                     : (EU_DIR + 'eu_copper/copper_map_fill.tif',                    'mg/kg',  (0.0,  None)),
    
    # Metales pesados — 1000 m (eu_heavy_metals)
    
    'eu_Ni'                     : (EU_DIR + 'eu_heavy_metals/Ni_EU27.tif',                      'mg/kg',  (0.0,  None)),
    'eu_Pb'                     : (EU_DIR + 'eu_heavy_metals/Pb_EU27.tif',                      'mg/kg',  (0.0,  None)),
    
    # Zinc — 1000 m (eu_zinc)

    'eu_Zn'                     : (EU_DIR + 'eu_zinc/zinc.tif',                                 'mg/kg', (0.0, None)),
    
    # Categóricas: Zonas Ambientales — 100 m (eu_env_zones_2018_esdac)
    # Códigos enteros 1-14; nodata=0 (uint8). Etiquetas en EEA_ENV_ZONES.
    
    'eu_env_zone'               : (EU_DIR + 'eu_env_zones_2018_esdac/eu_env_zones_2018_esdac.tif',          'código', (1, 14)),
    
    # Categóricas: Uso del Suelo — 100 m (eu_land_cover_2018_corine)
    # Códigos enteros 111-999; nodata=-128 (int8). Etiquetas en CORINE_LABELS.
    
    'eu_land_cover'             : (EU_DIR + 'eu_land_cover_2018_corine/eu_land_cover_2018_corine.tif',      'código', (100, 999)),
    
    # Categóricas: Tipo de Suelo WRB — 1000 m (eu_soil_type_wrb_2006)
    # Códigos enteros 1-30; nodata=0 (uint8). Etiquetas en WRB_SOIL_LABELS.
    # Valores 1-6 = no-suelo (Urbano, Agua, etc.); 7-30 = Grupos de Referencia WRB.
    
    'eu_soil_type_wrb'          : (EU_DIR + 'eu_soil_type_wrb_2006/eu_soil_type_wrb_2006_esdac.tif',        'código', (1, 30)),
    
    # Formato para añadir nuevas capas (si se obtienen nuevos datos)
    # 'eu_XXX': (EU_DIR + 'XXX.tif', 'unidad', (lim_inf, lim_sup)),
    # Al meter datos nuevos es recomendable ejecutar de neuvo desde la celda 28.
}

print(f'{len(EU_RASTERS)} capas raster europeas registradas')
for col, (path, unit, bounds) in EU_RASTERS.items():
    print(f'  {col:35s}  {unit:7s}  límites={bounds}')


20 capas raster europeas registradas
  eu_bulk_density                      g/cm³    límites=(0.0, 3.0)
  eu_clay_content                      %        límites=(0.0, 100.0)
  eu_sand_content                      %        límites=(0.0, 100.0)
  eu_silt_content                      %        límites=(0.0, 100.0)
  eu_soil_texture_class                USDA     límites=(1, 12)
  eu_water_holding_capacity            vol fr   límites=(0.0, None)
  eu_CN_ratio                          ratio    límites=(0.0, None)
  eu_K                                 mg/kg    límites=(0.0, None)
  eu_N                                 g/kg     límites=(0.0, None)
  eu_P                                 mg/kg    límites=(0.0, None)
  eu_pH                                pH       límites=(0.0, 14.0)
  eu_As                                %        límites=(0.0, 100.0)
  eu_organic_carbon_octop              %        límites=(0.0, 100.0)
  eu_Cu                                mg/kg    límites=(0.0, None)
  eu_Ni    

In [83]:
# ── Extracción vectorizada: abre cada raster UNA sola vez ───────────────────
# Rendimiento: 20 aperturas de archivo (una por capa) en lugar de 428×20=8560.
# rasterio.sample() lee solo los tiles necesarios — eficiente en GeoTIFF/COG.

lats = sites['latitude'].to_numpy()
lons = sites['longitude'].to_numpy()

eu_records = {}
for col, (path, unit, bounds) in tqdm(EU_RASTERS.items(),
                                       desc='Extracción rasters EU',
                                       total=len(EU_RASTERS)):
    eu_records[col] = batch_query_raster(path, lats, lons)

df_eu = pd.DataFrame(eu_records, index=sites.index)

# Añadir SITE_ID como columna para que el merge posterior funcione
df_eu.insert(0, 'SITE_ID', sites['SITE_ID'].values)

print(f'Características EU — forma: {df_eu.shape}')
df_eu.head(3)


Extracción rasters EU:   0%|          | 0/20 [00:00<?, ?it/s]

Características EU — forma: (428, 21)


,SITE_ID,eu_bulk_density,eu_clay_content,eu_sand_content,eu_silt_content,eu_soil_texture_class,eu_water_holding_capacity,eu_CN_ratio,eu_K,eu_N,eu_P,eu_pH,eu_As,eu_organic_carbon_octop,eu_Cu,eu_Ni,eu_Pb,eu_Zn,eu_env_zone,eu_land_cover,eu_soil_type_wrb
0,BE_001,1.0610,9.4843,61.2884,29.2273,12.0000,0.0877,12.3327,175.3472,3.6444,53.1383,5.6583,5.7471,3.2933,9.3738,16.7974,25.9992,42.9650,7.0000,18.0000,24.0000
1,BE_002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BE_003,1.0945,11.0963,59.4709,29.4328,12.0000,0.0815,12.4574,141.9444,2.8223,50.3453,5.7138,5.8829,3.2933,9.3738,16.7974,25.9992,45.6000,7.0000,18.0000,24.0000


In [84]:
# Columnas de etiqueta para capas categóricas
# Se añade *_nombre junto al código numérico; los NaN permanecen NaN.

def _add_label_col(df, code_col, lookup, label_col):
    if code_col not in df.columns:
        return
    df[label_col] = (
        df[code_col].dropna().astype(int).map(lookup)
        .reindex(df.index)   # realinear con el índice completo (NaN donde faltaba)
    )

_add_label_col(df_eu, 'eu_soil_texture_class', USDA_TEXTURE,    'eu_textura_suelo_nombre')
_add_label_col(df_eu, 'eu_env_zone',           EEA_ENV_ZONES,   'eu_zona_ambiental_nombre')
_add_label_col(df_eu, 'eu_land_cover',         CORINE_LABELS,   'eu_uso_suelo_nombre')
_add_label_col(df_eu, 'eu_soil_type_wrb',      WRB_SOIL_LABELS, 'eu_tipo_suelo_wrb_nombre')

for code_col, name_col in [
    ('eu_soil_texture_class', 'eu_textura_suelo_nombre'),
    ('eu_env_zone',           'eu_zona_ambiental_nombre'),
    ('eu_land_cover',         'eu_uso_suelo_nombre'),
    ('eu_soil_type_wrb',      'eu_tipo_suelo_wrb_nombre'),
]:
    if code_col not in df_eu.columns:
        continue
    counts = df_eu[name_col].value_counts(dropna=False)
    print(f'\n{name_col} ({counts.notna().sum()} sitios con valor):')
    print(counts.head(8).to_string())



eu_textura_suelo_nombre (9 sitios con valor):
eu_textura_suelo_nombre
Sandy Loam         113
Silt Loam           74
Loam                62
NaN                 56
Clay Loam           50
Silty Clay Loam     44
Sand                21
Loamy Sand           6

eu_zona_ambiental_nombre (10 sitios con valor):
eu_zona_ambiental_nombre
ATC    140
NaN     52
PAN     51
CON     43
ALS     37
LUS     33
NEM     27
ALN     24

eu_uso_suelo_nombre (1 sitios con valor):
eu_uso_suelo_nombre
NaN    428

eu_tipo_suelo_wrb_nombre (15 sitios con valor):
eu_tipo_suelo_wrb_nombre
CM – Cambisol    139
LV – Luvisol      73
NaN               53
PZ – Podzol       52
GL – Gleysol      19
PH – Phaeozem     18
Town              18
LP – Leptosol     15


In [85]:
# Informe de cobertura por capa 
cobertura = (df_eu.notna().sum() / len(df_eu) * 100).round(1)
cobertura = cobertura.rename('cobertura_%').to_frame()
cobertura['n_validos'] = df_eu.notna().sum()
cobertura['n_nulos']   = df_eu.isna().sum()

print('Cobertura por capa raster EU:')
print(cobertura.sort_values('cobertura_%').to_string())

# Capas con cobertura insuficiente (<80%) — revisar rutas o extensión geográfica
bajas = cobertura[cobertura['cobertura_%'] < 80]
if len(bajas):
    print(f'\n[WARN] {len(bajas)} capas con cobertura < 80%:')
    print(bajas.to_string())


Cobertura por capa raster EU:
                           cobertura_%  n_validos  n_nulos
eu_uso_suelo_nombre             0.0000          0      428
eu_CN_ratio                    72.7000        311      117
eu_N                           72.7000        311      117
eu_P                           72.7000        311      117
eu_pH                          72.7000        311      117
eu_K                           72.7000        311      117
eu_As                          75.7000        324      104
eu_Ni                          75.9000        325      103
eu_Pb                          75.9000        325      103
eu_Zn                          75.9000        325      103
eu_Cu                          76.2000        326      102
eu_soil_texture_class          86.9000        372       56
eu_textura_suelo_nombre        86.9000        372       56
eu_bulk_density                87.1000        373       55
eu_water_holding_capacity      87.1000        373       55
eu_clay_content           

In [86]:
# Verificación de límites de dominio y recorte 
# Aplica los límites físicos definidos en EU_RASTERS para detectar y corregir
# valores fuera de rango (e.g. pH > 14, arcilla > 100%).

flag_eu = []
for col, (path, unit, (lo, hi)) in EU_RASTERS.items():
    if col not in df_eu.columns:
        continue
    mask = pd.Series(False, index=df_eu.index)
    if lo is not None:
        mask |= df_eu[col] < lo
    if hi is not None:
        mask |= df_eu[col] > hi
    n = mask.sum()
    if n:
        flag_eu.append({'columna': col, 'unidad': unit, 'n_fuera_rango': n,
                        'rango_valido': f'[{lo}, {hi}]'})
        df_eu[col] = df_eu[col].clip(
            lower=lo if lo is not None else -np.inf,
            upper=hi if hi is not None else  np.inf
        )

if flag_eu:
    print('Valores fuera de rango detectados y recortados:')
    print(pd.DataFrame(flag_eu).to_string(index=False))
else:
    print('Todos los límites de dominio superados')


Valores fuera de rango detectados y recortados:
      columna unidad  n_fuera_rango rango_valido
eu_land_cover código            376   [100, 999]


---
## 10 · Fusión de Todas las Fuentes

Todos los datasets se unen a la tabla base `sites` mediante `SITE_ID` (left join — conserva los 428 sitios).


In [87]:
# Fusión de todas las fuentes sobre la tabla base de sitios 
# Left join en SITE_ID → conserva los 428 sitios aunque no tengan datos en
# alguna fuente (NaN en esas columnas, gestionados en la limpieza).

plan_fusion = [
    (phys_site,       'fisicos_sitio'),
    (phys_plot_pivot, 'fisicos_parcela'),
    (chem_site,       'quimicos_sitio'),
    (chem_plot_agg,   'quimicos_parcela'),
    (alpha_div,       'diversidad_alfa'),
    (micro_div,       'diversidad_microbiana'),
    (macro_agg,       'macrofauna'),
    (ew_com_agg,      'lombrices_combinado'),
    (nuid_site,       'lombrices_nuid'),
    (uvigo_site,      'lombrices_uvigo'),
    (orib_sum,        'oribatida'),
    (meso_sum,        'mesostigmata'),
    (coll_sum,        'colembola'),
    (bac_sum,         'bacterias_seq'),
    (fun_sum,         'hongos_seq'),
    (euk_sum,         'eucariotas_seq'),
    (oomy_sum,        'oomycetes_seq'),
    (cerc_sum,        'cercozoa_seq'),
    (df_eu,           'rasters_eu'),
]

df = sites.copy()
for right, etiqueta in plan_fusion:
    # Verificar que SITE_ID existe en el dataframe derecho
    if 'SITE_ID' not in right.columns:
        print(f' [WARN] {etiqueta}: sin columna SITE_ID — omitido')
        continue
    antes = df.shape[1]
    df = df.merge(right, on='SITE_ID', how='left', suffixes=('', f'_{etiqueta}'))
    print(f'  + {etiqueta:25s} → +{df.shape[1]-antes:4d} cols  (total: {df.shape[1]})')

print(f'\nForma total fusionada: {df.shape}')
# Verificación: no se deben haber perdido filas
assert len(df) == len(sites), f'¡Pérdida de filas! {len(sites)} → {len(df)}'
print('Integridad de filas verificada...')


  + fisicos_sitio             → +   6 cols  (total: 19)
  + fisicos_parcela           → +   5 cols  (total: 24)
  + quimicos_sitio            → +   9 cols  (total: 33)
  + quimicos_parcela          → +   3 cols  (total: 36)
  + diversidad_alfa           → +   8 cols  (total: 44)
  + diversidad_microbiana     → +   3 cols  (total: 47)
  + macrofauna                → +   2 cols  (total: 49)
  + lombrices_combinado       → + 114 cols  (total: 163)
  + lombrices_nuid            → +  28 cols  (total: 191)
  + lombrices_uvigo           → +   5 cols  (total: 196)
  + oribatida                 → +   2 cols  (total: 198)
  + mesostigmata              → +   2 cols  (total: 200)
  + colembola                 → +   2 cols  (total: 202)
  + bacterias_seq             → +   2 cols  (total: 204)
  + hongos_seq                → +   2 cols  (total: 206)
  + eucariotas_seq            → +   2 cols  (total: 208)
  + oomycetes_seq             → +   2 cols  (total: 210)
  + cercozoa_seq              → +   2 

---
## 11 · Limpieza de Datos

### 11.1 Auditoría de Valores Perdidos


In [88]:
missing = (
    df.isnull().sum()
      .rename('n_missing')
      .to_frame()
)
missing['pct_missing'] = (missing['n_missing'] / len(df) * 100).round(1)
missing = missing[missing['n_missing'] > 0].sort_values('pct_missing', ascending=False)

print(f'Total columns     : {df.shape[1]}')
print(f'Columns with NaNs : {len(missing)}')
print(f'Completely empty  : {(missing["pct_missing"] == 100).sum()}')
print()

# Show distribution of missingness
bins = [0, 5, 25, 50, 75, 100]
labels = ['<5%', '5–25%', '25–50%', '50–75%', '75–100%']
missing['bucket'] = pd.cut(missing['pct_missing'], bins=bins, labels=labels, right=True)
print('Missing value distribution:')
print(missing['bucket'].value_counts().sort_index())

Total columns     : 236
Columns with NaNs : 221
Completely empty  : 50

Missing value distribution:
bucket
<5%        26
5–25%      52
25–50%     47
50–75%     22
75–100%    74
Name: count, dtype: int64


In [89]:
# Eliminar columnas por encima del umbral de valores perdidos
MISS_THRESHOLD = 70.0   # %

drop_cols = missing[missing['pct_missing'] >= MISS_THRESHOLD].index.tolist()
print(f'Dropping {len(drop_cols)} columns (≥{MISS_THRESHOLD}% missing)')

df_clean = df.drop(columns=drop_cols)
print(f'Shape after drop: {df_clean.shape}')

Dropping 84 columns (≥70.0% missing)
Shape after drop: (428, 152)


In [90]:
# Eliminar filas sin coordenadas 
n_before = len(df_clean)
df_clean = df_clean.dropna(subset=['latitude', 'longitude'])
print(f'Dropped {n_before - len(df_clean)} rows with missing coordinates')

# Imputar NaN numéricos restantes con la mediana de la columna 
num_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())

# Imputar NaN categóricos con la moda 
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    mode_val = df_clean[col].mode()
    if len(mode_val) > 0:
        df_clean[col] = df_clean[col].fillna(mode_val[0])

remaining_na = df_clean.isnull().sum().sum()
print(f'Missing values remaining: {remaining_na}')

Dropped 1 rows with missing coordinates
Missing values remaining: 0


### 11.2 Comprobaciones de Sentido Físico


In [91]:
# Límites físicos para datos de suelo y medioambiente
DOMAIN_BOUNDS = {
    'latitude'           : (-90,    90),
    'longitude'          : (-180,  180),
    'clay_content'       : (0,     100),
    'silt_content'       : (0,     100),
    'sand_content'       : (0,     100),
    'Bulk density'       : (0,       3),   # g/cm³
    'Soil moisture'      : (0,       1),   # fraction
    'aggregate_stability': (0,       1),
    'soil_pH'            : (0,      14),
    'plot_Total_organic_C': (0,    100),   # %
    'plot_Total_N'       : (0,     100),   # %
    'Total_Plant_cover'  : (0,     100),   # %
    # Alpha diversity indices must be non-negative
    **{c: (0, None) for c in df_clean.columns if 'Shannon' in c or 'richness' in c.lower()},
}

flag_report = []
for col, (lo, hi) in DOMAIN_BOUNDS.items():
    if col not in df_clean.columns:
        continue
    mask = pd.Series([False] * len(df_clean), index=df_clean.index)
    if lo is not None:
        mask |= df_clean[col] < lo
    if hi is not None:
        mask |= df_clean[col] > hi
    n_out = mask.sum()
    if n_out:
        flag_report.append({'column': col, 'n_invalid': n_out, 'valid_range': f'[{lo}, {hi}]'})
        df_clean[col] = df_clean[col].clip(
            lower=lo if lo is not None else -np.inf,
            upper=hi if hi is not None else  np.inf
        )

if flag_report:
    print('Out-of-range values clipped:')
    print(pd.DataFrame(flag_report).to_string(index=False))
else:
    print('All domain checks passed')

Out-of-range values clipped:
             column  n_invalid valid_range
      Soil moisture         13      [0, 1]
aggregate_stability         36      [0, 1]
  Total_Plant_cover          1    [0, 100]
 Macrofauna_Shannon          2   [0, None]
  Earthworm_Shannon          2   [0, None]
 Collembola_Shannon          9   [0, None]


In [92]:
# Verificar suma de texturas: arcilla + limo + arena ≈ 100% 
tex = ['clay_content', 'silt_content', 'sand_content']
if all(c in df_clean.columns for c in tex):
    texture_sum = df_clean[tex].sum(axis=1)
    bad = ((texture_sum < 95) | (texture_sum > 105)).sum()
    print(f'Texture sum outside [95–105]%: {bad} rows')
    if bad:
        df_clean[tex] = df_clean[tex].div(texture_sum, axis=0).mul(100)
        print('  → Renormalised to 100%')

Texture sum outside [95–105]%: 40 rows
  → Renormalised to 100%


### 11.3 Detección de Outliers


In [93]:
def iqr_outlier_mask(series: pd.Series, factor: float = 3.0) -> pd.Series:
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return (series < Q1 - factor * IQR) | (series > Q3 + factor * IQR)


outlier_report = []
for col in df_clean.select_dtypes(include=np.number).columns:
    n = iqr_outlier_mask(df_clean[col]).sum()
    if n:
        outlier_report.append({'column': col, 'n_outliers': n,
                                'pct': round(n / len(df_clean) * 100, 1)})

df_outlier_rpt = (pd.DataFrame(outlier_report)
                    .sort_values('n_outliers', ascending=False))

print(f'Columns with IQR outliers (factor=3): {len(df_outlier_rpt)}')
print(df_outlier_rpt.head(20).to_string(index=False))

# Marcar filas que son outliers en CUALQUIER columna numérica
df_clean['outlier_flag'] = False
for col in df_clean.select_dtypes(include=np.number).columns:
    df_clean['outlier_flag'] |= iqr_outlier_mask(df_clean[col])

print(f'\nRows flagged as outliers: {df_clean["outlier_flag"].sum()} '
      f'({df_clean["outlier_flag"].mean()*100:.1f}%)')

Columns with IQR outliers (factor=3): 96
                        column  n_outliers     pct
nuid_Earthworm_biomass (fixed)         211 49.4000
          nuid_total_abundance         206 48.2000
                 nuid_Fragment         205 48.0000
                 nuid_Epilobic         200 46.8000
   nuid_Earthworm_spp_richness         189 44.3000
                nuid_Tanylobic         177 41.5000
 nuid_Allolobophora chlorotica          94 22.0000
       nuid_Aporrectodea rosea          93 21.8000
         ew_Aporrectodea longa          81 19.0000
  nuid_Aporrectodea caliginosa          71 16.6000
         ew_Lumbricus rubellus          71 16.6000
   ew_Allolobophora chlorotica          70 16.4000
       nuid_Aporrectodea longa          68 15.9000
         ew_Aporrectodea rosea          64 15.0000
   ew_Aporrectodea trapezoides          56 13.1000
    ew_Aporrectodea caliginosa          55 12.9000
nuid_Earthworm_biomass (fresh)          49 11.5000
            ew_total_abundance          4

### 11.4 Filas Duplicadas


In [94]:
n_exact = df_clean.duplicated().sum()
n_coord = df_clean.duplicated(subset=['latitude', 'longitude']).sum()

print(f'Exact duplicate rows       : {n_exact}')
print(f'Duplicate coordinates      : {n_coord}')

if n_exact:
    df_clean = df_clean.drop_duplicates()
    print(f'Dropped exact dupes. Shape: {df_clean.shape}')

Exact duplicate rows       : 0
Duplicate coordinates      : 37


---
## 12 · Estandarización y Codificación

### 12.1 Codificación de Variables Categóricas


In [95]:
from sklearn.preprocessing import LabelEncoder
import joblib

# Variables categóricas / nominales a codificar con LabelEncoder
# IMPORTANTE: eu_soil_texture_class, eu_env_zone y eu_soil_type_wrb son códigos
# NOMINALES (sin orden natural). Deben codificarse, NO escalarse con StandardScaler.
CATEGORICAL_COLS = [
    'Country',
    'Pedoclimatic_region',
    'soil_type',
    'Land_use_type',
    'Land_use_intensity',
    'Dominant_vegetation',
    'eu_soil_texture_class',  # USDA nominal 1-12
    'eu_env_zone',            # EEA zona nominal 1-15
    'eu_soil_type_wrb',       # WRB nominal 1-30
    'eu_land_cover',          # CORINE nominal compacto 1-44
]

label_encoders = {}
for col in CATEGORICAL_COLS:
    if col not in df_clean.columns:
        continue
    le = LabelEncoder()
    df_clean[col + '_enc'] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} clases → {col}_enc')

joblib.dump(label_encoders, OUT_DIR + 'label_encoders.pkl')
print(f'\nEncoders guardados → {OUT_DIR}label_encoders.pkl')


  Country: 12 clases → Country_enc
  Pedoclimatic_region: 9 clases → Pedoclimatic_region_enc
  soil_type: 22 clases → soil_type_enc
  Land_use_type: 7 clases → Land_use_type_enc
  Land_use_intensity: 3 clases → Land_use_intensity_enc
  Dominant_vegetation: 36 clases → Dominant_vegetation_enc
  eu_soil_texture_class: 8 clases → eu_soil_texture_class_enc
  eu_env_zone: 9 clases → eu_env_zone_enc
  eu_soil_type_wrb: 14 clases → eu_soil_type_wrb_enc
  eu_land_cover: 1 clases → eu_land_cover_enc

Encoders guardados → output/label_encoders.pkl


### 12.2 Estandarización Numérica (StandardScaler)


In [96]:
from sklearn.preprocessing import StandardScaler

# Columnas excluidas del escalado:
#   - Coordenadas e IDs (no son features)
#   - Variables categóricas nominales (ya codificadas con _enc)
#   - Indicadores _was_missing (son binarios 0/1, no escalar)
#   - _enc y _was_missing columnas
EXCLUDE_FROM_SCALING = [
    'latitude', 'longitude', 'SITE_ID', 'SAMPLE_ID', 'Sampling_date', 'outlier_flag',
    # Nominales codificadas — excluir versión cruda (la _enc sí se escala si se desea,
    # pero habitualmente se usa directamente como entero)
    'eu_soil_texture_class', 'eu_env_zone', 'eu_soil_type_wrb', 'eu_land_cover',
]

scale_cols = [
    c for c in df_clean.select_dtypes(include=np.number).columns
    if c not in EXCLUDE_FROM_SCALING
    and not c.endswith('_enc')
    and not c.endswith('_was_missing')   # binarios: no escalar
]

scaler = StandardScaler()
scaled_arr = scaler.fit_transform(df_clean[scale_cols])
df_scaled  = pd.DataFrame(
    scaled_arr,
    columns=[c + '_z' for c in scale_cols],
    index=df_clean.index
)

joblib.dump(scaler, OUT_DIR + 'scaler.pkl')
print(f'Escaladas {len(scale_cols)} columnas numéricas')
print(f'Scaler guardado: {OUT_DIR}scaler.pkl')


Escaladas 133 columnas numéricas
Scaler guardado: output/scaler.pkl


In [97]:
# Ensamblar el dataset final
df_final = pd.concat([df_clean.reset_index(drop=True),
                      df_scaled.reset_index(drop=True)], axis=1)

print(f'Final dataset: {df_final.shape[0]} sites × {df_final.shape[1]} columns')

Final dataset: 427 sites × 296 columns


### 12.3 Catálogo de Columnas


In [98]:
def infer_source(col):
    if col.startswith('eu_'):           return 'EU Raster'
    if col.startswith('macro_'):        return 'Macrofauna'
    if col.startswith('ew_'):           return 'Earthworms (combined)'
    if col.startswith('nuid_'):         return 'Earthworms (NUID_UCD raw)'
    if col.startswith('uvigo_'):        return 'Earthworms (UVIGO raw)'
    if col.startswith('orib_'):         return 'Oribatida'
    if col.startswith('meso_'):         return 'Mesostigmata'
    if col.startswith('coll_'):         return 'Collembola'
    if col.startswith('bac_'):          return 'Bacteria (16S)'
    if col.startswith('fun_'):          return 'Fungi (ITS)'
    if col.startswith('euk_'):          return 'Eukaryotes (18S)'
    if col.startswith('oomy_'):         return 'Oomycetes'
    if col.startswith('cerc_'):         return 'Cercozoa'
    if col.startswith('plot_'):         return 'Abiotic (plot-level)'
    if col in ['clay_content','silt_content','sand_content',
               'aggregate_stability','Bulk density','Soil moisture']: return 'Abiotic (physical)'
    if col in ['As','Cu','K','Mo','Ni','P','Pb','Zn','soil_pH']:      return 'Abiotic (chemical)'
    if 'Shannon' in col or 'SHANNON' in col:                          return 'Alpha diversity'
    if col.endswith('_z'):                                             return 'Scaled (z-score)'
    if col.endswith('_enc'):                                           return 'Encoded (label)'
    return 'Site metadata'

catalogue = pd.DataFrame({
    'dtype'     : df_final.dtypes,
    'n_missing' : df_final.isnull().sum(),
    'n_unique'  : df_final.nunique(),
    'source'    : [infer_source(c) for c in df_final.columns],
})

print('\nColumns by source:')
print(catalogue.groupby('source').size().sort_values(ascending=False).to_string())
catalogue


Columns by source:
source
Earthworms (combined)        72
Earthworms (NUID_UCD raw)    56
EU Raster                    43
Alpha diversity              22
Scaled (z-score)             16
Abiotic (plot-level)         16
Site metadata                14
Abiotic (chemical)            9
Abiotic (physical)            6
Encoded (label)               6
Collembola                    4
Cercozoa                      4
Bacteria (16S)                4
Eukaryotes (18S)              4
Fungi (ITS)                   4
Mesostigmata                  4
Macrofauna                    4
Oribatida                     4
Oomycetes                     4


,dtype,n_missing,n_unique,source
SITE_ID,object,0,427,Site metadata
SAMPLE_ID,object,0,427,Site metadata
Country,object,0,12,Site metadata
Pedoclimatic_region,object,0,9,Site metadata
Site_locality,object,0,181,Site metadata
...,...,...,...,...
eu_organic_carbon_octop_z,float64,0,122,EU Raster
eu_Cu_z,float64,0,239,EU Raster
eu_Ni_z,float64,0,210,EU Raster
eu_Pb_z,float64,0,210,EU Raster


---
## 13 · Exportación

Tres archivos de salida:
- `sob4es_clean.csv` — dataset limpio completo (escala original, legible)
- `sob4es_model_ready.csv` — solo columnas `_z` (escaladas) y `_enc` (codificadas), listo para ML
- `scaler.pkl` / `label_encoders.pkl` — transformadores para aplicar a nuevos datos


In [99]:
# 1. Limpio (legible por humanos) 
clean_path = OUT_DIR + 'sob4es_clean_v5.csv'
df_clean.to_csv(clean_path, index=False)
print(f'{clean_path}  ({df_clean.shape[0]} rows × {df_clean.shape[1]} cols) cleaned...')

# 2. Listo para modelo (escalado + codificado) 
id_cols    = ['SITE_ID', 'latitude', 'longitude', 'Country',
              'Pedoclimatic_region', 'Land_use_type', 'Sampling_date', 'outlier_flag']
model_cols = (
    id_cols
    + [c for c in df_final.columns if c.endswith('_z') or c.endswith('_enc')]
)
model_path = OUT_DIR + 'sob4es_model_ready_v5.csv'
df_final[[c for c in model_cols if c in df_final.columns]].to_csv(model_path, index=False)
print(f'{model_path}  ({df_final.shape[0]} rows × {len([c for c in model_cols if c in df_final.columns])} cols) ready...')

# 3. Informe de validación de lombrices 
check.to_csv(OUT_DIR + 'earthworm_validation_v5.csv', index=False)
print(f'{OUT_DIR}earthworm_validation.csv check done...')

print('\nData preparation complete...')
print(f'Final shape: {df_clean.shape[0]} sites × {df_clean.shape[1]} features')

output/sob4es_clean_v5.csv  (427 rows × 163 cols) cleaned...
output/sob4es_model_ready_v5.csv  (427 rows × 151 cols) ready...
output/earthworm_validation.csv check done...

Data preparation complete...
Final shape: 427 sites × 163 features
